<div style="background: linear-gradient(135deg, #4211BA, #33C6CD); padding: 24px; border-radius: 12px; color: white;">
<h1>🤖 Transformación de Variables: Encoding y Escalado</h1>
<h3>Módulo III — Machine Learning </h3>
</div>

## 🎯 Objetivos de la sesión

Al terminar esta clase serás capaz de:

1. **Explicar** por qué los algoritmos de Machine Learning no pueden trabajar directamente con texto y son sensibles a la escala de las variables.
2. **Diferenciar** cuándo usar **OneHot Encoding**, **Ordinal Encoding**, **Target Encoding**, **StandardScaler**, **MinMaxScaler** y **RobustScaler**.
3. **Aplicar** las seis técnicas en Python usando `scikit-learn` y `category_encoders`.
4. **Reconocer** ejemplos reales de la industria donde se usa cada técnica.

### ¿Por qué necesitamos transformar variables?

Los modelos de Machine Learning **solo entienden números**. Además, muchos algoritmos son sensibles a la escala de esos números.  
Este notebook resuelve dos problemas:

| Problema | Solución |
|---|---|
| Tengo columnas con texto/categorías | **Encoding** (convertir a número) |
| Mis columnas numéricas tienen escalas muy distintas | **Escalado** (poner todo en la misma escala) |

---

## 🔵 Bloque 1

## 🤔 ¿Por qué necesitamos "encoding"?

Los modelos de Machine Learning son, en el fondo, **operaciones matemáticas**. Solo entienden números.

Si una columna dice `"Grande"`, `"Madrid"` o `"Café"`, el modelo no sabe qué hacer con eso.

Vamos a comprobarlo.

In [ ]:
# 📦 Instalación de librerías necesarias (ejecutar solo si falta alguna)
# !pip install scikit-learn pandas numpy matplotlib seaborn category_encoders

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Mini ejemplo: intentamos entrenar un modelo con texto sin convertir
test_data = pd.DataFrame({
    'size': ['Small', 'Big', 'Medium'],
    'tip': [1.5, 3.0, 2.0]
})

modelo = LinearRegression()
try:
    modelo.fit(test_data[['size']], test_data['tip'])
except Exception as e:
    print("❌ Error:", e)

**Por eso existe el encoding**: convertir categorías de texto en números, pero de forma *inteligente*, según el tipo de información que representan.

Vamos a ver 3 formas distintas de hacerlo desde el Encoding

| Técnica | ¿Cuándo usarla? | Pros | Contras | Algoritmos que la prefieren |
| :--- | :--- | :--- | :--- | :--- |
| **Ordinal** | Categorías con orden natural (Bajo < Medio < Alto) | Simple, preserva jerarquía | Falso si no hay orden real | Todos (especialmente lineales) |
| **One-Hot** | Categorías nominales sin orden (Ciudad, Color) | No asume orden, interpretable | Alta dimensionalidad si hay muchas clases | Modelos lineales, KNN, SVM, Redes |
| **Target** | Variables con muchas categorías (>10-15) | Compacta, captura relación con target | Riesgo de data leakage si no se usa CV | Árboles, Lineales, Boosting |

## 📚 Las 3 técnicas de Encoding son:

---

### 1️⃣ OneHot Encoding (Grupo 1)
**Definición:** crea una columna binaria (0/1) por cada categoría. Ninguna categoría es "mayor" que otra.

**¿Cuándo usarla?** Cuando las categorías son **nominales** (sin orden) y son **pocas** (idealmente menos de 10-15).

**Ejemplo en la vida real:**
- Tipo de transacción (compra / devolución / transferencia) en un modelo de detección de fraude.
- Color de un producto en un sistema de recomendación de e-commerce.
- Método de pago (tarjeta / efectivo / transferencia) en un modelo de riesgo de impago.

---

### 2️⃣ Ordinal Encoding (Grupo 2)
**Definición:** asigna un número que respeta un **orden lógico** entre las categorías (0, 1, 2, 3...).

**¿Cuándo usarla?** Cuando las categorías **sí tienen jerarquía o secuencia natural**.

**Ejemplo en la vida real:**
- Nivel educativo (primaria < secundaria < universidad) en un modelo de aprobación de crédito.
- Severidad de un ticket de soporte (bajo / medio / alto / crítico) para priorizar atención.
- Talla de ropa (S < M < L < XL) en un modelo de gestión de inventario.

---

### 3️⃣ Target Encoding (Grupo 3)
**Definición:** reemplaza cada categoría por un número calculado a partir de la **variable objetivo (target)** — normalmente el promedio del target para esa categoría.

**¿Cuándo usarla?** Cuando hay **muchas categorías distintas** (alta cardinalidad) y OneHot generaría demasiadas columnas.

**Ejemplo en la vida real:**
- Código postal o ciudad en un modelo de predicción de precios de vivienda.
- ID de tienda en un modelo de predicción de ventas con miles de tiendas.
- ID de vendedor en un modelo de probabilidad de cierre de venta (CRM).

> ⚠️ **Dato importante:** como esta técnica usa la variable objetivo, es un encoding *supervisado* — y por eso requiere más cuidado (riesgo de **fuga de datos / data leakage**). Lo veremos en detalle más adelante en el módulo. Hoy nos enfocamos en entender la idea.

## ☕ Nuestro caso: Pedidos de una Cafetería

Imaginemos que tenemos los datos de 16 pedidos. Vamos a identificar juntos qué tipo de encoding le corresponde a cada columna categórica.

In [ ]:
data = {
    'order_id': list(range(1, 17)),
    'size':   ['Small','Big','Medium','Big','Small','Medium','Big','Small',
                 'Medium','Big','Small','Medium','Big','Small','Medium','Big'],
    'drink':   ['Coffee','Tea','Coffee','Chocolate','Juice','Tea',  'Coffee',       'Coffee','Chocolate','Juice','Tea','Coffee','Coffee',   'Chocolate','Juice','Tea'],
    'city':   ['Madrid','Barcelona','Madrid','Valencia','Sevilla','Barcelona',        'Madrid','Sevilla','Valencia','Madrid','Barcelona','Sevilla','Madrid','Valencia','Barcelona','Sevilla'],
    'tip':  [1.50, 3.00, 2.00, 2.50, 1.00, 2.20, 3.50, 1.20,
                 2.80, 2.00, 1.50, 1.80, 3.20, 1.40, 2.10, 2.60]
}

df = pd.DataFrame(data)
df

---
# GRUPO 1

---
## 1️⃣ OneHot Encoding — columna `bebida`

`bebida` no tiene orden (Café no es "mayor" que Té), así que cada categoría se convierte en su propia columna binaria.

¿Cuándo usarlo?

  Variables categóricas nominales (sin orden jerárquico)
  Ej: Color, Ciudad, Género

  
**Concepto clave:** Crea una columna binaria (0/1) por cada categoría.

### 🧐 Pregunta:

Mirando el dataframe, ¿qué columna(s) categórica(s) tienen:
- ¿Un **orden natural**? →
- ¿Pocas categorías **sin orden**? →
- ¿**Muchas** categorías repetidas, sin orden? →

In [ ]:
#Importamos la herramienta de scikit-learn que hace el OneHot Encoding por nosotros.
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)

drink_encoded = ohe.fit_transform(df[['drink']])
df_ohe = pd.DataFrame(drink_encoded, columns=ohe.get_feature_names_out(['drink']))
df_ohe


### 🧐 Pregunta guiada



1.   ¿Por qué **no** simplemente asignamos Café=1, Té=2, Chocolate=3, Jugo=4?

>          *Pista: piensa en qué entendería el modelo si Jugo (4) es "el doble" de Té (2).*

2.   ¿Por qué es importante dentro del Machine Learning?





### 💭 Para reflexionar

Si tuvieran una columna `país` con 190 categorías distintas en un dataset de e-commerce, ¿usarían esta técnica y por qué?

Escribe aquí tu respuesta:


---



---
 # GRUPO 2

---

### 🧐 Pregunta:

Mirando el dataframe, ¿qué columna(s) categórica(s) tienen:
- ¿Un **orden natural**? →
- ¿Pocas categorías **sin orden**? →
- ¿**Muchas** categorías repetidas, sin orden? →

----
## 2️⃣ Ordinal Encoding — columna `tamaño`

`tamaño` SÍ tiene un orden lógico: Pequeño < Mediano < Grande. Aquí una sola columna numérica es suficiente.

¿Cuándo usarlo?

    Variables categóricas ordinales (con orden jerárquico)
    Ej: Nivel de estudios, Rating, Talla

In [ ]:
#Importamos la herramienta de scikit-learn para hacer Ordinal Encoding.
from sklearn.preprocessing import OrdinalEncoder

order_sizes = [['Small', 'Medium', 'Big']]

oe = OrdinalEncoder(categories=order_sizes)

df['size_encoded'] = oe.fit_transform(df[['size']])
df

### 🧐 Preguntas



1.   ¿Por qué aquí SÍ podemos usar 0, 1, 2... y en `bebida` no?
2.   ¿Por qué es importante dentro del Machine Learning?


### 💭 Para reflexionar

Si tuvieran una columna `país` con 190 categorías distintas en un dataset de e-commerce, ¿usarían esta técnica y por qué?


---
# GRUPO 3

---

### 🧐 Pregunta:

Mirando el dataframe, ¿qué columna(s) categórica(s) tienen:
- ¿Un **orden natural**? →
- ¿Pocas categorías **sin orden**? →
- ¿**Muchas** categorías repetidas, sin orden? →

---
## 3️⃣ Target Encoding — columna `ciudad`

`ciudad` tiene varias categorías y no queremos crear demasiadas columnas binarias. En su lugar, reemplazamos cada ciudad por el **promedio de propina** en esa ciudad.


¿Cuándo usarlo?

   Variables categóricas con alta cardinalidad (muchas categorías)
   Ej: Código postal, Marca de coche, ID de usuario

**Concepto clave:** Reemplaza cada categoría por la media del target para esa categoría.

In [ ]:
#Instalamos category_encoders
!pip install category_encoders -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.1 MB/s eta 0:00:00


In [ ]:
import category_encoders as ce

te = ce.TargetEncoder()

df['city_encoded'] = te.fit_transform(df['city'], df['tip'])

df[['city', 'city_encoded', 'tip']].sort_values('city')

### 🧐 Pregunta guiada




1.   ¿Por qué es importante dentro del Machine Learning?
2.   ¿De dónde sale ese número para cada ciudad? ¿Por qué Madrid tiene un valor distinto al de Sevilla?

>       *Pista: vuelve a mirar la columna `propina` para cada ciudad.*

### 💭 Para reflexionar

Si tuvieran una columna `país` con 190 categorías distintas en un dataset de e-commerce, ¿usarían esta técnica y por qué?

---
## ✅ Resumen: ¿cuál técnica uso cuándo?

| Técnica | ¿Hay orden? | ¿Cuántas categorías? | Usa el target? | Ejemplo real |
|---|---|---|---|---|
| **OneHot** | No | Pocas (<15) | No | Color de producto, tipo de pago |
| **Ordinal** | Sí | Cualquier cantidad | No | Nivel educativo, severidad de ticket |
| **Target** | No | Muchas (alta cardinalidad) | Sí | Código postal, ID de tienda |



🔗 INTEGRACIÓN CON ALGORITMOS

| Tipo de Modelo | ¿Necesita Encoding? | ¿Necesita Escalado? | Notas |
| :--- | :--- | :---: | :--- |
| **Lineal/Logístico, Ridge, Lasso** | Sí (One-Hot/Target) | ✅ Sí | Escalado ayuda a interpretar coeficientes y converge más rápido |
| **KNN, SVM, PCA, Redes Neuronales** | Sí | ✅ Sí | Basados en distancias/gradients. Sin escalado, *Fare* dominará a *Age* |
| **Árboles, Random Forest, XGBoost** | Sí (Ordinal/One-Hot o nativo) | ❌ No | Dividen por umbrales, la escala no afecta splits |
| **Regresión (predicción numérica)** | Igual | ✅ Sí | Ridge/Lasso penalizan coeficientes; escalado hace penalización justa |



📌 Regla de oro: Si el modelo usa distancias, productos punto o regularización → escala. Si usa divisiones por umbrales → no escala.


--------------------
## 🟢 Bloque 2

-------------------

## 🤔 ¿Por qué necesitamos "escalado"?

Muchos algoritmos de Machine Learning (KNN, K-Means, regresión logística, redes neuronales) calculan **distancias** entre datos. Si una columna tiene números mucho más grandes que otra, esa columna "domina" la distancia sin que eso tenga sentido real.

Vamos a comprobarlo.

In [ ]:
import numpy as np

# Two employees: [age, annual_salary]
employee_1 = np.array([25, 28000])
employee_2 = np.array([52, 48000])

distance = np.linalg.norm(employee_1 - employee_2)
print("Distance between employees:", distance)

### 🧐 Pregunta guiada

¿Es justo que la diferencia de **edad** (27 años) casi no influya en esta distancia, solo porque el **salario** se mide en una escala mucho más grande?

## 📚 Las 3 técnicas de Escalado son:

---

### 1️⃣ StandardScaler (Grupo 4)
**Definición:** transforma los datos para que tengan **media 0** y **desviación estándar 1**. Fórmula: `(x - media) / desviación_estándar`.

**¿Cuándo usarla?** Cuando los datos no tienen outliers extremos y siguen una distribución más o menos normal (forma de campana).

**Ejemplo en la vida real:**
- Estandarizar altura y peso de pacientes en un modelo de diagnóstico médico.
- Variables de entrada en una red neuronal o un SVM.

---

### 2️⃣ MinMaxScaler (Grupo 5)
**Definición:** transforma los datos para que queden en un rango fijo, normalmente entre 0 y 1. Fórmula: `(x - mínimo) / (máximo - mínimo)`.

**¿Cuándo usarla?** Cuando necesitas un rango acotado específico y **no hay outliers extremos** (un solo valor muy alto o muy bajo "aplasta" a todos los demás cerca de 0).

**Ejemplo en la vida real:**
- Normalizar píxeles de una imagen (0-255 → 0-1) para una red neuronal de visión por computadora.
- Variables de entrada en sistemas de recomendación.

---

### 3️⃣ RobustScaler (Todos)
**Definición:** usa la **mediana** y el **rango intercuartílico (IQR)** en lugar de la media y la desviación estándar. Fórmula: `(x - mediana) / IQR`.

**¿Cuándo usarla?** Cuando el dataset tiene valores atípicos que no quieres eliminar, pero que distorsionarían a las otras dos técnicas.

**Ejemplo en la vida real:**
- Salarios o ingresos (siempre hay sueldos excepcionalmente altos, como el de un CEO).
- Precios de propiedades de lujo en un dataset inmobiliario.
- Montos de transacciones financieras con compras excepcionalmente grandes.

## 🏢 Nuestro caso: Empleados de una empresa

Tenemos los datos de 12 empleados: edad y salario anual. Uno de ellos es el CEO, con un salario mucho más alto que el resto.

In [ ]:
import pandas as pd

data = {
    'employee_id': list(range(1, 13)),
    'age': [25, 34, 45, 29, 52, 38, 41, 26, 33, 58, 30, 47],
    'annual_salary': [28000, 35000, 42000, 31000, 48000, 39000,
                       45000, 29000, 34000, 250000, 32000, 44000]
}

df = pd.DataFrame(data)
df

### 🧐 Pregunta:


Mirando el dataframe, ¿qué empleado se sale claramente del patrón del resto? ¿En qué columna?

---
## 1️⃣ StandardScaler (Grupo 4)

Vamos a transformar `age` y `annual_salary` para que ambas tengan media 0 y desviación estándar 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
df_standard = pd.DataFrame(
    standard_scaler.fit_transform(df[['age', 'annual_salary']]),
    columns=['age_standard', 'salary_standard']
)

pd.concat([df[['employee_id', 'age', 'annual_salary']], df_standard], axis=1)

### 🧐 Preguntas:

1. Observa el valor escalado del CEO (fila 10). ¿Qué tan extremo se ve comparado con los demás?
2. ¿Por qué es importante dentro del Machine Learning esta técnica?

---
## 2️⃣ MinMaxScaler (Grupo 5)

Ahora transformamos las mismas columnas para que queden entre 0 y 1.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(df[['age', 'annual_salary']]),
    columns=['age_minmax', 'salary_minmax']
)

pd.concat([df[['employee_id', 'age', 'annual_salary']], df_minmax], axis=1)

### 🧐 Pregunta guiada

1. Mira la columna `salary_minmax`. ¿Qué les pasó a los demás empleados (sin contar al CEO)? ¿Por qué quedaron todos tan apretados, cerca de 0?
2. ¿Por qué es importante dentro del Machine Learning esta técnica?

---
## 3️⃣ RobustScaler (Todos)

Esta vez usamos la mediana y el IQR en lugar de la media y la desviación estándar — pensado justo para casos con outliers como el del CEO.

In [ ]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
df_robust = pd.DataFrame(
    robust_scaler.fit_transform(df[['age', 'annual_salary']]),
    columns=['age_robust', 'salary_robust']
)

pd.concat([df[['employee_id', 'age', 'annual_salary']], df_robust], axis=1)

### 🧐 Pregunta guiada

Compara `salary_robust` con `salary_minmax` para los empleados que NO son el CEO.
1. ¿Cuál técnica conserva mejor las diferencias reales entre ellos?
2. ¿Por qué es importante dentro del Machine Learning esta técnica?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(df['employee_id'], df_standard['salary_standard'])
axes[0].set_title('StandardScaler')

axes[1].bar(df['employee_id'], df_minmax['salary_minmax'])
axes[1].set_title('MinMaxScaler')

axes[2].bar(df['employee_id'], df_robust['salary_robust'])
axes[2].set_title('RobustScaler')

plt.suptitle('Salary scaled by each technique (employee 10 = CEO)')
plt.tight_layout()
plt.show()

---
## ✅ Resumen: ¿cuál técnica uso cuándo?

| Técnica | Usa | ¿Sensible a outliers? | Ejemplo real |
|---|---|---|---|
| **StandardScaler** | Media y desviación estándar | Sí | Altura/peso en diagnóstico médico, SVM |
| **MinMaxScaler** | Mínimo y máximo | Muy sensible | Píxeles de imágenes, redes neuronales |
| **RobustScaler** | Mediana e IQR | No (por diseño) | Salarios, precios inmobiliarios |



### 💭 Para reflexionar

Si estuvieran preparando datos de transacciones de tarjeta de crédito para un modelo de detección de fraude (donde unas pocas transacciones son extremadamente grandes), ¿qué técnica usarían y por qué?

📚 **Resumen y Conclusiones**

In [ ]:
# 📚 RESUMEN FINAL
print("📚 RESUMEN FINAL: TRANSFORMACIÓN DE VARIABLES")
print("="*60)

resumen = """
## 🔑 CONCEPTOS CLAVE

### 1. Encoding de Variables Categóricas
- **One-Hot Encoding (OHE)**: Para variables nominales sin orden
  - Usa `pd.get_dummies()` para análisis rápido
  - Usa `sklearn.preprocessing.OneHotEncoder` para producción

- **Ordinal Encoding**: Para variables con orden jerárquico
  - Define explícitamente el orden de las categorías
  - Asegúrate de que el orden tenga sentido para el negocio

- **Target Encoding**: Para variables con alta cardinalidad
  - Reemplaza categorías por la media del target
  - ¡CUIDADO CON EL DATA LEAKAGE!
  - Usa `sklearn.preprocessing.TargetEncoder` (sklearn 1.3+)

### 2. Escalado de Variables Numéricas
- **StandardScaler**: Media=0, Std=1 (para datos sin outliers)
- **MinMaxScaler**: Rango [0, 1] (para redes neuronales)
- **RobustScaler**: Usa mediana e IQR (robusto a outliers)

### 3. Buenas Prácticas
- ✅ Siempre hacer train_test_split ANTES de preprocesar
- ✅ Usar ColumnTransformer para diferentes tipos de variables
- ✅ Usar Pipelines para encadenar transformaciones
- ✅ Serializar (pickle/joblib) el pipeline completo
- ✅ Manejar categorías nuevas en producción

### 4. Errores Comunes
- ❌ Data Leakage: Usar información del test en train
- ❌ Aplicar fit en test (usar transform)
- ❌ No manejar categorías nuevas en producción
- ❌ Usar OHE para variables ordinales
- ❌ No escalar features para modelos basados en distancias

## 🎯 CUÁNDO USAR CADA TÉCNICA

| Situación | Técnica Recomendada | Alternativa |
|-----------|---------------------|-------------|
| Variable nominal baja cardinalidad | OneHotEncoder | get_dummies |
| Variable ordinal | OrdinalEncoder | Mapeo manual |
| Variable alta cardinalidad | TargetEncoder | Mean encoding |
| Datos sin outliers | StandardScaler | MinMaxScaler |
| Datos con outliers | RobustScaler | QuantileTransformer |
| Redes neuronales | MinMaxScaler | StandardScaler |
| Modelos lineales | StandardScaler | RobustScaler |

"""

print(resumen)

# 📊 Tabla resumen de técnicas
import pandas as pd

tabla_resumen = pd.DataFrame({
    'Técnica': ['OneHotEncoder', 'OrdinalEncoder', 'TargetEncoder',
                'StandardScaler', 'MinMaxScaler', 'RobustScaler'],
    'Tipo': ['Categórico', 'Categórico', 'Categórico',
             'Numérico', 'Numérico', 'Numérico'],
    'Cuándo usar': ['Nominal baja cardinalidad', 'Ordinal', 'Alta cardinalidad',
                    'Sin outliers', 'Redes neuronales', 'Con outliers'],
    'Sklearn class': ['OneHotEncoder', 'OrdinalEncoder', 'TargetEncoder',
                      'StandardScaler', 'MinMaxScaler', 'RobustScaler'],
    'Cuidado con': ['Multicolinealidad', 'Orden incorrecto', 'Data leakage',
                    'Outliers', 'Outliers', 'Ninguno especial']
})

print("\n📊 TABLA RESUMEN DE TÉCNICAS:")
display(tabla_resumen)


📚 RESUMEN FINAL: TRANSFORMACIÓN DE VARIABLES

## 🔑 CONCEPTOS CLAVE

### 1. Encoding de Variables Categóricas
- **One-Hot Encoding (OHE)**: Para variables nominales sin orden
  - Usa `pd.get_dummies()` para análisis rápido
  - Usa `sklearn.preprocessing.OneHotEncoder` para producción

- **Ordinal Encoding**: Para variables con orden jerárquico
  - Define explícitamente el orden de las categorías
  - Asegúrate de que el orden tenga sentido para el negocio

- **Target Encoding**: Para variables con alta cardinalidad
  - Reemplaza categorías por la media del target
  - ¡CUIDADO CON EL DATA LEAKAGE!
  - Usa `sklearn.preprocessing.TargetEncoder` (sklearn 1.3+)

### 2. Escalado de Variables Numéricas
- **StandardScaler**: Media=0, Std=1 (para datos sin outliers)
- **MinMaxScaler**: Rango [0, 1] (para redes neuronales)
- **RobustScaler**: Usa mediana e IQR (robusto a outliers)

### 3. Buenas Prácticas
- ✅ Siempre hacer train_test_split ANTES de preprocesar
- ✅ Usar ColumnTransformer para difer

,Técnica,Tipo,Cuándo usar,Sklearn class,Cuidado con
0,OneHotEncoder,Categórico,Nominal baja cardinalidad,OneHotEncoder,Multicolinealidad
1,OrdinalEncoder,Categórico,Ordinal,OrdinalEncoder,Orden incorrecto
2,TargetEncoder,Categórico,Alta cardinalidad,TargetEncoder,Data leakage
3,StandardScaler,Numérico,Sin outliers,StandardScaler,Outliers
4,MinMaxScaler,Numérico,Redes neuronales,MinMaxScaler,Outliers
5,RobustScaler,Numérico,Con outliers,RobustScaler,Ninguno especial


📖 **Recursos Adicionales**

In [ ]:
# 📚 RECURSOS ADICIONALES
print("📚 RECURSOS ADICIONALES PARA PROFUNDIZAR")
print("="*60)

recursos = """
## 📚 Documentación Oficial
- [Scikit-learn: Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Scikit-learn: ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)
- [Scikit-learn: Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)

## 📖 Tutoriales Recomendados
- [ColumnTransformer with Mixed Types](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)
- [How to Use the ColumnTransformer for Data Preparation](https://www.machinelearningmastery.com/columntransformer-for-numerical-and-categorical-data)
- [Target Encoding Implementation](https://codefinity.com/blog/Target-Encoding-Implementation)

## 🎥 Videos Educativos
- [How to use Sklearn ColumnTransformer](https://www.youtube.com/watch?v=lPYaxjBiKGE)
- [Compare the effect of different scalers on data with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html)

## 🔧 Librerías Complementarias
- [category_encoders](https://github.com/scikit-learn-contrib/category_encoders): Más opciones de encoding
- [feature-engine](https://feature-engine.readthedocs.io/): Transformaciones específicas

## 📊 Datasets para Practicar
- [Titanic Dataset](https://www.kaggle.com/c/titanic): Variables categóricas y numéricas
- [House Prices Dataset](https://www.kaggle.com/c/house-prices-advanced-regression-techniques): Muchas variables categóricas
- [Adult Income Dataset](https://archive.ics.uci.edu/ml/datasets/adult): Clásico para preprocessing
"""

print(recursos)

📚 RECURSOS ADICIONALES PARA PROFUNDIZAR

## 📚 Documentación Oficial
- [Scikit-learn: Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Scikit-learn: ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)
- [Scikit-learn: Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)

## 📖 Tutoriales Recomendados
- [ColumnTransformer with Mixed Types](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)
- [How to Use the ColumnTransformer for Data Preparation](https://www.machinelearningmastery.com/columntransformer-for-numerical-and-categorical-data)
- [Target Encoding Implementation](https://codefinity.com/blog/Target-Encoding-Implementation)

## 🎥 Videos Educativos
- [How to use Sklearn ColumnTransformer](https://www.youtube.com/watch?v=lPYaxjBiKGE)
- [Compare the effect of different scalers on data with outliers](https://scikit-lea